# Final MARBERT models (Intent + Sentiment) — train, save, and test for an app

**What this notebook produces**
- `saudi_marbert_intent/` and `saudi_marbert_sentiment/`: two saved models (weights + tokenizer + label names), each ~650 MB.
- `training_summary.json`
- A ready-to-use `SaudiClassifier` class: give it an Arabic message, get back the intent and the sentiment with confidence scores.

**Before you run:** `Runtime > Change runtime type > T4 GPU`, and have `Clean_saudi_NLP.csv` ready to upload.

**Protocol = the same one we evaluated in cross-validation:** each model trains on 90% of the 308 messages and keeps the epoch that scores best on the other 10%.
So the honest expectation for new messages is the **CV numbers** (Intent F1 ~0.89, Sentiment F1 ~0.90), NOT the validation score printed here (that one is optimistic because the epoch was chosen on it).

**Important for the app:** the models were trained on *cleaned* text. The `clean_text` function below reproduces that cleaning
(it matches the `text_clean` column on all 308 rows), and `SaudiClassifier.predict` applies it automatically.

In [ ]:
!pip install -q transformers

In [ ]:
import os, re, json, random, time
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
if device.type == "cpu":
    print("WARNING: no GPU. Runtime > Change runtime type > T4 GPU.")

MODEL_NAME = "UBC-NLP/MARBERT"
MAX_LEN    = 32
BATCH_SIZE = 16
EPOCHS     = 10
LR_RATE    = 2e-5
SEED       = 42
TASKS      = ["intent", "sentiment"]

# ---- The SAME cleaning that produced the text_clean column (verified on all 308 rows below) ----
_DIACRITICS = re.compile(r"[\u064B-\u0652\u0670\u0640]")   # tanween / harakat / tatweel

def clean_text(t):
    t = str(t)
    t = _DIACRITICS.sub("", t)                 # remove diacritics and tatweel
    t = re.sub(r"[أإآ]", "ا", t)               # normalise alef forms
    t = re.sub(r"(.)\1{2,}", r"\1\1", t)        # 3+ repeated chars 
    t = re.sub(r"['\"“”‘’]", "", t)           # remove quote marks
    t = re.sub(r"\s+", " ", t).strip()         # collapse spaces
    return t

In [ ]:
while not os.path.exists("Clean_saudi_NLP.csv"):
    from google.colab import files
    print("Upload Clean_saudi_NLP.csv")
    files.upload()

df = pd.read_csv("Clean_saudi_NLP.csv")
print("rows:", len(df))

# Safety check: our clean_text must reproduce the training text exactly
match = (df["text"].map(clean_text) == df["text_clean"]).sum()
print(f"clean_text reproduces text_clean on {match}/{len(df)} rows")
assert match == len(df), "clean_text does not match text_clean -> do not deploy until fixed"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def encode_texts(texts, y_ids):
    enc = tokenizer(list(texts), padding="max_length", truncation=True,
                    max_length=MAX_LEN, return_tensors="pt")
    return TensorDataset(enc["input_ids"], enc["attention_mask"], torch.tensor(y_ids, dtype=torch.long))

def predict_ids(model, loader):
    model.eval()
    preds = []
    with torch.no_grad():
        for ids, mask, _ in loader:
            logits = model(input_ids=ids.to(device), attention_mask=mask.to(device)).logits
            preds.extend(logits.argmax(dim=-1).cpu().tolist())
    return np.array(preds)

In [ ]:
def train_final(task):
    set_seed(SEED)
    labels = sorted(df[task].unique())
    label2id = {l: i for i, l in enumerate(labels)}
    id2label = {i: l for l, i in label2id.items()}
    y = df[task].map(label2id).values
    texts = df["text_clean"].astype(str).values

    tr_idx, va_idx = train_test_split(np.arange(len(df)), test_size=0.10, stratify=y, random_state=SEED)
    g = torch.Generator(); g.manual_seed(SEED)
    train_loader = DataLoader(encode_texts(texts[tr_idx], y[tr_idx]), batch_size=BATCH_SIZE, shuffle=True, generator=g)
    val_loader   = DataLoader(encode_texts(texts[va_idx], y[va_idx]), batch_size=64)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=len(labels), id2label=id2label, label2id=label2id).to(device)

    counts = np.bincount(y[tr_idx], minlength=len(labels))
    weights = torch.tensor(len(tr_idx) / (len(labels) * np.maximum(counts, 1)), dtype=torch.float).to(device)
    loss_fn = torch.nn.CrossEntropyLoss(weight=weights)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR_RATE, weight_decay=0.01)
    total_steps = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)

    best_f1, best_epoch, best_state = -1.0, -1, None
    for epoch in range(1, EPOCHS + 1):
        model.train()
        for ids, mask, yb in train_loader:
            optimizer.zero_grad()
            logits = model(input_ids=ids.to(device), attention_mask=mask.to(device)).logits
            loss = loss_fn(logits, yb.to(device))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
        pred = predict_ids(model, val_loader)
        f1 = f1_score(y[va_idx], pred, average="macro", zero_division=0)
        print(f"  [{task}] epoch {epoch:2d} | val F1-macro = {f1:.3f}")
        if f1 > best_f1:
            best_f1, best_epoch = f1, epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    pred = predict_ids(model, val_loader)
    out_dir = f"saudi_marbert_{task}"
    model.save_pretrained(out_dir)
    tokenizer.save_pretrained(out_dir)

    summary = {"task": task, "labels": labels, "best_epoch": best_epoch,
               "n_train": int(len(tr_idx)), "n_val": int(len(va_idx)),
               "val_f1_macro_OPTIMISTIC": round(float(best_f1), 4),
               "val_accuracy_OPTIMISTIC": round(float(accuracy_score(y[va_idx], pred)), 4),
               "saved_to": out_dir}
    del model, optimizer, best_state
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return summary

In [ ]:
# Trains both models (a few minutes on a T4; the first run also downloads MARBERT)
summaries = []
for task in TASKS:
    t0 = time.time()
    print(f"=== {task} ===")
    s = train_final(task)
    summaries.append(s)
    print(f"  saved -> {s['saved_to']} | best epoch {s['best_epoch']} | ({time.time() - t0:.0f}s)")

with open("training_summary.json", "w", encoding="utf-8") as f:
    json.dump(summaries, f, ensure_ascii=False, indent=2)
print(json.dumps(summaries, ensure_ascii=False, indent=2))
print("\nNote: val scores above are optimistic (the epoch was chosen on them). Expect roughly the CV numbers on new messages.")

In [ ]:
class SaudiClassifier:
    """Loads the two saved models and classifies raw Arabic (Saudi dialect) messages."""

    def __init__(self, intent_dir="saudi_marbert_intent", sentiment_dir="saudi_marbert_sentiment",
                 review_threshold=0.60, device=None):
        self.device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))
        self.threshold = review_threshold
        self.tok = AutoTokenizer.from_pretrained(intent_dir)
        self.models, self.labels = {}, {}
        for task, path in [("intent", intent_dir), ("sentiment", sentiment_dir)]:
            m = AutoModelForSequenceClassification.from_pretrained(path).to(self.device).eval()
            self.models[task] = m
            self.labels[task] = {int(k): v for k, v in m.config.id2label.items()}

    def _probs(self, task, texts):
        enc = self.tok(texts, padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt").to(self.device)
        with torch.no_grad():
            logits = self.models[task](**enc).logits
        return torch.softmax(logits, dim=-1).cpu().numpy()

    def predict(self, texts):
        single = isinstance(texts, str)
        if single:
            texts = [texts]
        cleaned = [clean_text(t) for t in texts]
        results = [None] * len(texts)
        keep = [i for i, c in enumerate(cleaned) if c]          # skip empty messages
        if keep:
            batch = [cleaned[i] for i in keep]
            p_int, p_sen = self._probs("intent", batch), self._probs("sentiment", batch)
            for row, i in enumerate(keep):
                ki, ks = int(p_int[row].argmax()), int(p_sen[row].argmax())
                ci, cs = float(p_int[row][ki]), float(p_sen[row][ks])
                results[i] = {"text": texts[i], "cleaned": cleaned[i],
                              "intent": self.labels["intent"][ki], "intent_confidence": round(ci, 3),
                              "sentiment": self.labels["sentiment"][ks], "sentiment_confidence": round(cs, 3),
                              "needs_human_review": bool(min(ci, cs) < self.threshold)}
        for i in range(len(texts)):
            if results[i] is None:
                results[i] = {"text": texts[i], "cleaned": "", "intent": None, "sentiment": None,
                              "needs_human_review": True}
        return results[0] if single else results

In [ ]:
# Smoke test on NEW messages (written by hand, not from the dataset). Look at them yourself -- this is not an evaluation.
clf = SaudiClassifier()
samples = [
    "الطلب وصلني متأخر واجد وابغى ارجع فلوسي",
    "كيف اقدر اغير رقم جوالي بالحساب؟",
    "ما قدرت ادفع بالفيزا يطلع لي خطأ",
    "شكرا لكم الخدمة كانت سريعة وممتازة",
    "التطبيق يهنق كل ما افتحه",
    "اقترح تضيفون خيار الدفع بابل باي",
    "ابغى الغي اشتراكي",
]
out = pd.DataFrame(clf.predict(samples))
pd.set_option("display.max_colwidth", 60); pd.set_option("display.width", 250)
out[["text", "intent", "intent_confidence", "sentiment", "sentiment_confidence", "needs_human_review"]]

In [ ]:
# Save the models. RECOMMENDED: copy to Google Drive (a 1.3 GB browser download can fail).
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/saudi_nlp_models
!cp -r saudi_marbert_intent saudi_marbert_sentiment training_summary.json /content/drive/MyDrive/saudi_nlp_models/
!ls -la /content/drive/MyDrive/saudi_nlp_models/

### Next step: the app
- The saved folders load with `AutoModelForSequenceClassification.from_pretrained(folder)` (label names are stored inside).
- Keep `clean_text` and `SaudiClassifier` from this notebook: the app must clean text the same way.
- The `needs_human_review` flag (confidence < 0.60) is an arbitrary starting threshold; tune it on real messages.
- Limits to remember: trained on only 308 messages, expected accuracy ~0.89-0.90 (CV), and it may struggle with phrasing very different from the training data.